In [1]:
import os
import mysql.connector
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [2]:
load_dotenv()

def get_connection():

    connection = mysql.connector.connect(
        host=os.getenv("DB_HOST"),
        port=int(os.getenv("DB_PORT", 3306)),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        database=os.getenv("DB_NAME")
    )

    return connection

In [3]:
connection = get_connection()

query = "select * from eta_analysis_dataset;"

eta_df = pd.read_sql(query, connection)

eta_df.head()

,order_id,store_id,zone,store_name,order_timestamp,total_items,fresh_item_count,picking_capacity,rider_capacity,traffic_level,weather_condition,actual_delivery_minutes,promised_delivery_mintes,distance_km
0,ord_000001,str_000032,HSR Extension,QuickMart Whitefield Main Road,2026-01-01 00:03:00,2,0,10,12,Normal,Clear\r,11.0,14.0,1.73
1,ord_000002,str_000029,BTM Layout,QuickMart HSR Extension,2026-01-01 00:07:00,4,0,13,12,Normal,Clear\r,11.0,12.0,1.11
2,ord_000003,str_000001,Malleshwaram,QuickMart Koramangala,2026-01-01 00:11:00,2,1,14,17,Normal,Clear\r,11.0,13.0,1.51
3,ord_000004,str_000038,Kalyan Nagar,QuickMart Nagarbhavi,2026-01-01 00:11:00,3,0,21,20,Normal,Clear\r,9.0,11.0,1.15
4,ord_000005,str_000011,Jayanagar,QuickMart Hebbal,2026-01-01 00:12:00,6,2,8,11,High,Clear\r,13.0,15.0,1.01


In [4]:
avg_del_time = eta_df.actual_delivery_minutes.mean()
avg_del_time

np.float64(14.925746135085495)

In [5]:
eta_df.groupby(['fresh_item_count'])['actual_delivery_minutes'].mean()

fresh_item_count
0     14.663465
1     15.014913
2     15.407800
3     15.686097
4     16.106886
5     16.570923
6     16.958431
7     17.387263
8     18.028269
9     18.031496
10    19.405405
11    18.000000
12    19.210526
13    19.800000
14    20.000000
Name: actual_delivery_minutes, dtype: float64

In [6]:
eta_df.groupby(['picking_capacity'])['actual_delivery_minutes'].mean()

picking_capacity
7     17.180599
8     16.845670
9     16.699278
10    15.066817
11    15.893557
12    14.913874
13    14.561980
14    14.254492
15    14.223702
16    14.280568
17    13.606277
19    13.567264
21    13.477090
22    13.392411
Name: actual_delivery_minutes, dtype: float64

In [7]:
eta_df.groupby(['rider_capacity'])['actual_delivery_minutes'].mean()

rider_capacity
8     17.211404
10    16.813337
11    16.624637
12    15.131815
13    15.104996
14    14.429056
15    14.373498
16    14.166238
17    14.217454
18    14.281780
19    14.423818
20    13.955154
21    13.606277
24    13.392411
26    13.567264
Name: actual_delivery_minutes, dtype: float64

In [8]:
eta_df.groupby('weather_condition')['actual_delivery_minutes'].agg(['mean', 'median'])

,mean,median
weather_condition,,
Clear\r,14.292319,14.0
Heavy Rain\r,17.130162,17.0
Rain\r,16.334981,16.0


In [9]:
eta_df['weather_condition'] = eta_df['weather_condition'].str.replace('\r',"")

In [10]:
eta_df.head()

,order_id,store_id,zone,store_name,order_timestamp,total_items,fresh_item_count,picking_capacity,rider_capacity,traffic_level,weather_condition,actual_delivery_minutes,promised_delivery_mintes,distance_km
0,ord_000001,str_000032,HSR Extension,QuickMart Whitefield Main Road,2026-01-01 00:03:00,2,0,10,12,Normal,Clear,11.0,14.0,1.73
1,ord_000002,str_000029,BTM Layout,QuickMart HSR Extension,2026-01-01 00:07:00,4,0,13,12,Normal,Clear,11.0,12.0,1.11
2,ord_000003,str_000001,Malleshwaram,QuickMart Koramangala,2026-01-01 00:11:00,2,1,14,17,Normal,Clear,11.0,13.0,1.51
3,ord_000004,str_000038,Kalyan Nagar,QuickMart Nagarbhavi,2026-01-01 00:11:00,3,0,21,20,Normal,Clear,9.0,11.0,1.15
4,ord_000005,str_000011,Jayanagar,QuickMart Hebbal,2026-01-01 00:12:00,6,2,8,11,High,Clear,13.0,15.0,1.01


In [11]:
eta_df["order_timestamp"] = pd.to_datetime(eta_df["order_timestamp"])

In [12]:
eta_df['hour'] = eta_df.order_timestamp.dt.hour

In [13]:
counts = eta_df.hour.value_counts()
counts

hour
20    30846
19    28217
21    25879
18    21538
12    18599
11    17594
13    17551
10    16130
22    15971
14    15011
17    14732
9     14665
15    12264
8     12079
16    11371
7      6825
23     6017
6      3277
0      2714
1      1914
2      1575
5      1565
4      1311
3      1262
Name: count, dtype: int64

In [14]:
eta_df['is_peak_hour'] = eta_df['hour'].apply(lambda X:counts[X] >20000)

In [15]:
peak_hour_dict = eta_df.groupby("is_peak_hour")["hour"].unique().to_dict()

print("True hours:", peak_hour_dict[True])
print("False hours:", peak_hour_dict[False])

True hours: [18 19 20 21]
False hours: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 22 23]


In [16]:
eta_df['day_name']=eta_df['order_timestamp'].dt.day_name()

In [17]:
eta_df["day_name"].value_counts()

day_name
Saturday     48102
Friday       45088
Sunday       44043
Thursday     43102
Wednesday    40242
Tuesday      40025
Monday       38305
Name: count, dtype: int64

In [18]:
eta_df['is_weekend'] =eta_df['day_name'].isin(['Saturday','Sunday'])
eta_df.sample(10)

,order_id,store_id,zone,store_name,order_timestamp,total_items,fresh_item_count,picking_capacity,rider_capacity,traffic_level,weather_condition,actual_delivery_minutes,promised_delivery_mintes,distance_km,hour,is_peak_hour,day_name,is_weekend
6981,ord_007003,str_000003,Marathahalli,QuickMart BTM Layout,2026-01-05 07:04:00,3,1,13,12,Very High,Clear,13.0,15.0,1.00,7,False,Monday,False
41195,ord_041332,str_000031,Sarjapur Road,QuickMart Bellandur ORR,2026-01-25 18:02:00,7,2,12,12,Very High,Rain,20.0,25.0,0.82,18,True,Sunday,True
181730,ord_182399,str_000025,Koramangala,QuickMart Ulsoor,2026-04-21 14:52:00,5,3,11,11,High,Clear,16.0,15.0,0.95,14,False,Tuesday,False
58202,ord_058386,str_000009,HSR Extension,QuickMart Jayanagar,2026-02-04 22:39:00,2,0,12,12,Normal,Clear,13.0,12.0,1.01,22,False,Wednesday,False
264289,ord_265253,str_000023,Whitefield,QuickMart Vijayanagar,2026-06-10 20:31:00,2,0,19,26,Very High,Heavy Rain,25.0,28.0,2.71,20,True,Wednesday,False
40656,ord_040792,str_000034,Koramangala,QuickMart Jayanagar 4th Block,2026-01-25 11:45:00,6,1,9,10,Normal,Rain,14.0,15.0,0.61,11,False,Sunday,True
76471,ord_076724,str_000010,Sarjapur Road,QuickMart JP Nagar,2026-02-15 20:14:00,1,0,17,21,Very High,Rain,13.0,13.0,0.40,20,True,Sunday,True
208743,ord_209521,str_000039,Bellandur,QuickMart Kengeri,2026-05-08 08:34:00,3,1,9,8,High,Clear,16.0,19.0,1.56,8,False,Friday,False
12007,ord_012044,str_000018,BTM Layout,QuickMart CV Raman Nagar,2026-01-08 11:31:00,1,0,16,12,High,Clear,10.0,14.0,1.08,11,False,Thursday,False
293207,ord_294287,str_000037,Kengeri,QuickMart Yeshwanthpur,2026-06-27 18:57:00,4,1,16,19,Normal,Clear,13.0,13.0,0.75,18,True,Saturday,True


In [19]:
eta_df.groupby(["hour"])["actual_delivery_minutes"].mean()

hour
0     12.183493
1     12.310867
2     12.286984
3     12.285261
4     12.188406
5     12.237700
6     13.070186
7     14.097289
8     14.796092
9     14.826321
10    14.834780
11    13.464590
12    14.344965
13    14.299584
14    14.323363
15    13.106083
16    13.155219
17    13.913861
18    16.935277
19    16.869582
20    16.925274
21    16.873372
22    12.545050
23    12.543626
Name: actual_delivery_minutes, dtype: float64

In [20]:
eta_df.to_parquet('cleaned_eta.parquet', engine='pyarrow', index=False)

In [21]:
eta_df.columns

Index(['order_id', 'store_id', 'zone', 'store_name', 'order_timestamp',
       'total_items', 'fresh_item_count', 'picking_capacity', 'rider_capacity',
       'traffic_level', 'weather_condition', 'actual_delivery_minutes',
       'promised_delivery_mintes', 'distance_km', 'hour', 'is_peak_hour',
       'day_name', 'is_weekend'],
      dtype='str')